# Fashion mnist pytorch

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Pytorch/Fasion%20MNIST%20dataset%20using%20pytorch.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.optim as  optim
import torch.nn as nn

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
! pip install opendatasets

Download and extract the dataset described in `docs/DATASETS.md` before running the model cells. Notebook-specific credential uploads and mounted-drive setup have been removed.

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(str(TABULAR / 'fashion-mnist_train.csv'))

In [ ]:
data.head()


In [ ]:
torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
torch.manual_seed(42)

In [ ]:
fig , axes  = plt.subplots(4,4,figsize = (10,10))
fig.suptitle("First 16 images"  ,fontsize = 16)

for i , ax in enumerate(axes.flat):
  img = data.iloc[i , 1:].values.reshape(28,28)
  ax.imshow(img)
  ax.axis('off')
  ax.set_title(f"Label {data.iloc[i,0]}")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
X = data.iloc[: , 1:].values
y = data.iloc[: , 0].values



In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [ ]:
X_train = X_train / 255
X_test = X_test/255

In [ ]:
class CustomDataset(Dataset):
  def __init__(self ,features, labels):
    self.features = torch.tensor(features , dtype = torch.float32)
    self.labels = torch.tensor(labels , dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self , idx):
    return self.features[idx] , self.labels[idx]

In [ ]:


train_dataset  = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test , y_test)

In [ ]:
import torch.nn as nn

class NN(nn.Module):
    def __init__(self, num_features):
        super(NN, self).__init__()

        layers = []
        input_dim = num_features
        hidden_dim = 120  # Neurons per hidden layer
        num_hidden_layers = 5

        # Input Layer
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(0.1))

        # Hidden Layers
        for _ in range(num_hidden_layers - 1):  # 5 hidden layers total
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))

        # Output Layer (Assuming 10 output classes)
        layers.append(nn.Linear(hidden_dim, 10))

        # Combine all layers into a Sequential model
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)



In [ ]:
train_loader = DataLoader(train_dataset , batch_size = 16 , shuffle = True ,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size = 16 , shuffle = False , pin_memory=True)

In [ ]:
model = NN(X_train.shape[1])
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters() , lr = 0.008672666599580775 , weight_decay= 0.00022408559780089356)


In [ ]:
history = []
for epoch in range(epochs):
  epoch_loss = 0
  for batch_features , batch_labels in train_loader:
    batch_features = batch_features.to('cuda' if torch.cuda.is_available() else 'cpu')
    batch_labels = batch_labels.to('cuda' if torch.cuda.is_available() else 'cpu')
    optimizer.zero_grad()
    outputs = model(batch_features)
    loss = criterion(outputs , batch_labels)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()
  avg_loss = epoch_loss / len(train_loader)
  history.append(avg_loss)
  print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

In [ ]:
plt.plot(range(1, epochs+1), history, marker='o', linestyle='-')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss History')
plt.grid()
plt.show()


In [ ]:
model.eval()

In [ ]:
total = 0
correct = 0
with torch.no_grad():
  for batch_features , batch_labels in test_loader:
    batch_features = batch_features.to('cuda' if torch.cuda.is_available() else 'cpu')
    batch_labels = batch_labels.to('cuda' if torch.cuda.is_available() else 'cpu')
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
total = 0
correct = 0
with torch.no_grad():
  for batch_features , batch_labels in train_loader:
    batch_features = batch_features.to('cuda' if torch.cuda.is_available() else 'cpu')
    batch_labels = batch_labels.to('cuda' if torch.cuda.is_available() else 'cpu')
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Train Accuracy: {accuracy * 100:.2f}%")